In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
import hydra
from omegaconf import OmegaConf
from models.components.adapted_lm import AdaptedLM
from models.components.codestral import Codestral
import os

os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["PROJECT_ROOT"] = "/home/gerel/Documents/gramm-score"


/home/gerel/miniconda3/envs/gramscore/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path

dir = Path("../configs").absolute()
with initialize(version_base=None, config_path="../configs"):
    # cfg = compose("model/components/language_model/codestral.yaml")#, overrides=["+model_path=/share/homes/gerel/gramscore/weights/Codestral-22B-v0.1"])
    cfg = compose("train")
    print(OmegaConf.to_yaml(cfg))

task_name: split_asap
tags:
- dev
data:
  _target_: src.data.mnist_datamodule.MNISTDataModule
  data_dir: ${paths.data_dir}
  batch_size: 128
  train_val_test_split:
  - 55000
  - 5000
  - 10000
  num_workers: 0
  pin_memory: false
model:
  lm:
    config:
      _target_: transformers.BitsAndBytesConfig
      load_in_4bit: true
      bnb_4bit_use_double_quant: true
      bnb_4bit_quant_type: nf4
      bnb_4bit_compute_dtype: bfloat16
      bnb_4bit_quant_storage: bfloat16
    _target_: transformers.AutoModelForCausalLM.from_pretrained
    pretrained_model_name_or_path: mistralai/Codestral-22B-v0.1
    device_map: auto
  peft_config:
    _target_: peft.LoraConfig
    r: 64
    target_modules:
    - q_proj
    - k_proj
    - v_proj
    - o_proj
    - gate_proj
    - up_proj
    - down_proj
    - lm_head
    lora_alpha: 16
    lora_dropout: 0.1
    bias: none
    task_type: CAUSAL_LM
  _target_: src.models.components.adapted_lm.AdaptedLM
callbacks:
  model_checkpoint:
    _target_: lightn

In [4]:
import torch
import gc

try:
    del model
except:
    pass
torch.cuda.empty_cache()
gc.collect()

3394

In [5]:
cfg.model

{'lm': {'config': {'_target_': 'transformers.BitsAndBytesConfig', 'load_in_4bit': True, 'bnb_4bit_use_double_quant': True, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_compute_dtype': 'bfloat16', 'bnb_4bit_quant_storage': 'bfloat16'}, '_target_': 'transformers.AutoModelForCausalLM.from_pretrained', 'pretrained_model_name_or_path': 'mistralai/Codestral-22B-v0.1', 'device_map': 'auto'}, 'peft_config': {'_target_': 'peft.LoraConfig', 'r': 64, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'lm_head'], 'lora_alpha': 16, 'lora_dropout': 0.1, 'bias': 'none', 'task_type': 'CAUSAL_LM'}, '_target_': 'src.models.components.adapted_lm.AdaptedLM'}

In [6]:
# peft = hydra.utils.instantiate(cfg.model.components.peft_config)
# peft

In [7]:
model: AdaptedLM = hydra.utils.instantiate(cfg.model)
model

Loading checkpoint shards: 100%|██████████| 9/9 [00:12<00:00,  1.36s/it]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


AdaptedLM(
  (lm): MistralForCausalLM(
    (model): MistralModel(
      (embed_tokens): Embedding(32768, 6144)
      (layers): ModuleList(
        (0-55): 56 x MistralDecoderLayer(
          (self_attn): MistralSdpaAttention(
            (q_proj): lora.Linear(
              (base_layer): Linear(in_features=6144, out_features=6144, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=6144, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_features=6144, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (k_proj): lora.Linear(
              (base_layer): Linear(in_features=6144, out_features=1024, b

```
Codestral(
  (model): MistralForCausalLM(
    (model): MistralModel(
      (embed_tokens): Embedding(32768, 6144)
      (layers): ModuleList(
        (0-55): 56 x MistralDecoderLayer(
          (self_attn): MistralSdpaAttention(
            (q_proj): Linear(in_features=6144, out_features=6144, bias=False)
            (k_proj): Linear(in_features=6144, out_features=1024, bias=False)
            (v_proj): Linear(in_features=6144, out_features=1024, bias=False)
            (o_proj): Linear(in_features=6144, out_features=6144, bias=False)
            (rotary_emb): MistralRotaryEmbedding()
          )
          (mlp): MistralMLP(
            (gate_proj): Linear(in_features=6144, out_features=16384, bias=False)
            (up_proj): Linear(in_features=6144, out_features=16384, bias=False)
            (down_proj): Linear(in_features=16384, out_features=6144, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): MistralRMSNorm((6144,), eps=1e-05)
          (post_attention_layernorm): MistralRMSNorm((6144,), eps=1e-05)
        )
      )
      (norm): MistralRMSNorm((6144,), eps=1e-05)
    )
    (lm_head): Linear(in_features=6144, out_features=32768, bias=False)
  )
)
```
---
```
AdaptedLM(
  (lm): MistralForCausalLM(
    (model): MistralModel(
      (embed_tokens): Embedding(32768, 6144)
      (layers): ModuleList(
        (0-55): 56 x MistralDecoderLayer(
          (self_attn): MistralSdpaAttention(
            (q_proj): lora.Linear(
              (base_layer): Linear(in_features=6144, out_features=6144, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=6144, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_features=6144, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (k_proj): lora.Linear(
              (base_layer): Linear(in_features=6144, out_features=1024, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=6144, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_features=1024, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (v_proj): lora.Linear(
              (base_layer): Linear(in_features=6144, out_features=1024, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=6144, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_features=1024, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (o_proj): lora.Linear(
              (base_layer): Linear(in_features=6144, out_features=6144, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=6144, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_features=6144, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (rotary_emb): MistralRotaryEmbedding()
          )
          (mlp): MistralMLP(
            (gate_proj): lora.Linear(
              (base_layer): Linear(in_features=6144, out_features=16384, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=6144, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_features=16384, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (up_proj): lora.Linear(
              (base_layer): Linear(in_features=6144, out_features=16384, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=6144, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_features=16384, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (down_proj): lora.Linear(
              (base_layer): Linear(in_features=16384, out_features=6144, bias=False)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.1, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=16384, out_features=64, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=64, out_features=6144, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()
            )
            (act_fn): SiLU()
          )
          (input_layernorm): MistralRMSNorm((6144,), eps=1e-05)
          (post_attention_layernorm): MistralRMSNorm((6144,), eps=1e-05)
        )
      )
      (norm): MistralRMSNorm((6144,), eps=1e-05)
    )
    (lm_head): lora.Linear(
      (base_layer): Linear(in_features=6144, out_features=32768, bias=False)
      (lora_dropout): ModuleDict(
        (default): Dropout(p=0.1, inplace=False)
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=6144, out_features=64, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=64, out_features=32768, bias=False)
      )
      (lora_embedding_A): ParameterDict()
      (lora_embedding_B): ParameterDict()
      (lora_magnitude_vector): ModuleDict()
    )
  )
  (model): PeftModelForCausalLM(
    (base_model): LoraModel(
      (model): MistralForCausalLM(
        (model): MistralModel(
          (embed_tokens): Embedding(32768, 6144)
          (layers): ModuleList(
            (0-55): 56 x MistralDecoderLayer(
              (self_attn): MistralSdpaAttention(
                (q_proj): lora.Linear(
                  (base_layer): Linear(in_features=6144, out_features=6144, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=6144, out_features=64, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=64, out_features=6144, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict()
                )
                (k_proj): lora.Linear(
                  (base_layer): Linear(in_features=6144, out_features=1024, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=6144, out_features=64, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=64, out_features=1024, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict()
                )
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=6144, out_features=1024, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=6144, out_features=64, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=64, out_features=1024, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict()
                )
                (o_proj): lora.Linear(
                  (base_layer): Linear(in_features=6144, out_features=6144, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=6144, out_features=64, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=64, out_features=6144, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict()
                )
                (rotary_emb): MistralRotaryEmbedding()
              )
              (mlp): MistralMLP(
                (gate_proj): lora.Linear(
                  (base_layer): Linear(in_features=6144, out_features=16384, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=6144, out_features=64, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=64, out_features=16384, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict()
                )
                (up_proj): lora.Linear(
                  (base_layer): Linear(in_features=6144, out_features=16384, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=6144, out_features=64, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=64, out_features=16384, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict()
                )
                (down_proj): lora.Linear(
                  (base_layer): Linear(in_features=16384, out_features=6144, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=16384, out_features=64, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=64, out_features=6144, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (lora_magnitude_vector): ModuleDict()
                )
                (act_fn): SiLU()
              )
              (input_layernorm): MistralRMSNorm((6144,), eps=1e-05)
              (post_attention_layernorm): MistralRMSNorm((6144,), eps=1e-05)
            )
          )
          (norm): MistralRMSNorm((6144,), eps=1e-05)
        )
        (lm_head): lora.Linear(
          (base_layer): Linear(in_features=6144, out_features=32768, bias=False)
          (lora_dropout): ModuleDict(
            (default): Dropout(p=0.1, inplace=False)
          )
          (lora_A): ModuleDict(
            (default): Linear(in_features=6144, out_features=64, bias=False)
          )
          (lora_B): ModuleDict(
            (default): Linear(in_features=64, out_features=32768, bias=False)
          )
          (lora_embedding_A): ParameterDict()
          (lora_embedding_B): ParameterDict()
          (lora_magnitude_vector): ModuleDict()
        )
      )
    )
  )
)
```

In [8]:
from transformers import TextStreamer

In [9]:
print(OmegaConf.to_yaml(cfg.model))

lm:
  config:
    _target_: transformers.BitsAndBytesConfig
    load_in_4bit: true
    bnb_4bit_use_double_quant: true
    bnb_4bit_quant_type: nf4
    bnb_4bit_compute_dtype: bfloat16
    bnb_4bit_quant_storage: bfloat16
  _target_: transformers.AutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: mistralai/Codestral-22B-v0.1
  device_map: auto
peft_config:
  _target_: peft.LoraConfig
  r: 64
  target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
  - lm_head
  lora_alpha: 16
  lora_dropout: 0.1
  bias: none
  task_type: CAUSAL_LM
_target_: src.models.components.adapted_lm.AdaptedLM



In [23]:
import transformers
import re


tokenizer = transformers.AutoTokenizer.from_pretrained(cfg.model.lm.pretrained_model_name_or_path)

# Get the chat template, and encapsulate the text between
# "{%- elif message['role'] == 'assistant' %}" and "{%- else %}" in between
# "{% generation %}" and "{% endgeneration %}"
# See PR https://github.com/huggingface/transformers/pull/30650
template = tokenizer.chat_template
print(template)
template = re.sub(
    r"{%- elif message\['role'\] == 'assistant' %}(.+?){%- else %}",
    r"{%- elif message['role'] == 'assistant' %}{% generation %}\1{% endgeneration %}{%- else %}",
    template,
    flags=re.DOTALL,
)
print("###")
print(template)
tokenizer.chat_template = template


tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are an automatic music transcription assistant. Given a sequence of notes representing a performance, you will generate a score in the ABC+ format, and nothing else."},
        {"role": "user", "content": """Note(pitch=60, start=0.5, duration=0.848)
Note(pitch=62, start=1.095, duration=0.734)
Note(pitch=64, start=1.706, duration=0.749)
Note(pitch=65, start=2.365, duration=1.012)
Note(pitch=67, start=3.387, duration=0.443)
Note(pitch=65, start=3.527, duration=0.339)
Note(pitch=64, start=3.663, duration=0.853)"""},
        {"role": "assistant", "content": "ABC"},
        {"role": "user", "content": "Generate a Lilypond file of 'Jingle Bells'"},
        {"role": "assistant", "content": "Some Lilypond output to test this"},
    ],
    # tokenize=False,
    return_assistant_tokens_mask=True,
    return_dict=True,
)

{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content'] %}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set loop_messages = messages %}
{%- endif %}

{{- bos_token }}
{%- for message in loop_messages %}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}
        {{- raise_exception('After the optional system message, conversation roles must alternate user/assistant/user/assistant/...') }}
    {%- endif %}
    {%- if message['role'] == 'user' %}
        {%- if loop.last and system_message is defined %}
            {{- '[INST] ' + system_message + '\n\n' + message['content'] + '[/INST]' }}
        {%- else %}
            {{- '[INST] ' + message['content'] + '[/INST]' }}
        {%- endif %}
    {%- elif message['role'] == 'assistant' %}
        {{- ' ' + message['content'] + eos_token}}
    {%- else %}
        {{- raise_exception('Only user and assistant roles are supported, with the exception of an initial optional

{'input_ids': [1, 3, 6864, 29500, 29488, 2754, 29514, 29552, 29502, 29493, 1917, 29514, 29502, 29491, 29550, 29493, 12732, 29514, 29502, 29491, 29551, 29549, 29551, 29499, 781, 12973, 29500, 29488, 2754, 29514, 29552, 29518, 29493, 1917, 29514, 29508, 29491, 29502, 29542, 29550, 29493, 12732, 29514, 29502, 29491, 29555, 29538, 29549, 29499, 781, 12973, 29500, 29488, 2754, 29514, 29552, 29549, 29493, 1917, 29514, 29508, 29491, 29555, 29502, 29552, 29493, 12732, 29514, 29502, 29491, 29555, 29549, 29542, 29499, 781, 12973, 29500, 29488, 2754, 29514, 29552, 29550, 29493, 1917, 29514, 29518, 29491, 29538, 29552, 29550, 29493, 12732, 29514, 29508, 29491, 29502, 29508, 29518, 29499, 781, 12973, 29500, 29488, 2754, 29514, 29552, 29555, 29493, 1917, 29514, 29538, 29491, 29538, 29551, 29555, 29493, 12732, 29514, 29502, 29491, 29549, 29549, 29538, 29499, 781, 12973, 29500, 29488, 2754, 29514, 29552, 29550, 29493, 1917, 29514, 29538, 29491, 29550, 29518, 29555, 29493, 12732, 29514, 29502, 29491, 2

In [10]:

text = "Hello my name is"
# text = "Generate a Lilypond file of 'Jingle Bells'"
inputs = tokenizer(text, return_tensors="pt")

streamer = TextStreamer(tokenizer)
outputs = model.model.generate(**inputs, max_new_tokens=2, temperature=0.3, do_sample=False, streamer=streamer)

NameError: name 'tokenizer' is not defined


```
\version "2.22.1"

\header {
  title = "Jingle Bells"
  composer = ""
}

\score {
  \new Staff {
    \key c \major
    \time 4/4
    \tempo "Moderato"

    \relative c' {
      \repeat volta 2 {
        c8 d e d c d e g g a g f e d c
        c8 d e d c d e g g a g f e d c
        e8 e e d d d e g g g a a g f e d
        c8 d e d c d e g g a g f e d c
      }
    }
  }

  \layout { }
}
```